In [5]:
import os

def scan_files(root_dir=".", extensions=(".py",), exclude_dirs=None):
    if exclude_dirs is None:
        exclude_dirs = []

    # 🔥 Normalize excluded paths
    exclude_dirs = [os.path.normpath(os.path.join(root_dir, d)) for d in exclude_dirs]

    for dirpath, dirnames, filenames in os.walk(root_dir):
        # 🔥 Remove excluded directories based on FULL PATH
        dirnames[:] = [
            d for d in dirnames
            if os.path.normpath(os.path.join(dirpath, d)) not in exclude_dirs
        ]

        for filename in filenames:
            if filename.endswith(extensions):
                full_path = os.path.join(dirpath, filename)

                print("=" * 80)
                print(f"FILE: {full_path}")
                print("=" * 80)

                try:
                    with open(full_path, "r", encoding="utf-8") as f:
                        print(f.read())
                except Exception as e:
                    print(f"[ERROR] Could not read file: {e}")


if __name__ == "__main__":
    extensions_to_scan = (".py", ".json", ".md", ".txt")
    root_directory = "."

    # ✅ Use forward slashes (cross-platform safe)
    excluded_folders = [
        "benchmark-results-raw",
        "benchmarks",
        "docs/threshold-ablation-logs",
        "docs/papers"
    ]

    scan_files(root_directory, extensions_to_scan, excluded_folders)

FILE: .\PLAN.md
# TurboQuant Implementation Plan

> Implementing TurboQuant KV cache compression from scratch based on the ICLR 2026 paper (arXiv 2504.19874).
> Goal: a working Python/NumPy prototype that can compress and decompress KV cache tensors, validate correctness against the paper's distortion bounds, and eventually integrate with llama.cpp or MLX.

## Paper Reference

- **Title:** TurboQuant: Online Vector Quantization with Near-optimal Distortion Rate
- **Authors:** Amir Zandieh, Majid Daliri, Majid Hadian, Vahab Mirrokni
- **Venue:** ICLR 2026
- **arXiv:** 2504.19874
- **Source:** https://research.google/blog/turboquant-redefining-ai-efficiency-with-extreme-compression/

---

## Architecture Overview

TurboQuant is three algorithms composed together:

```
Input vector x ∈ R^d
        │
        ▼
┌─────────────────┐
│   PolarQuant     │  Random rotation → Beta-distributed coordinates
│   (b-1 bits)     │  → optimal scalar quantization per coordinate
└────────┬────────┘
      